# 涨跌停情绪因子：分组回测（连续因子有效性检验）

用 `backtest_timeseries_factor` 替代原阈值二值信号择时逻辑，
直接对连续因子做分位数分组回测，检验因子值高低分组间的区分力。

**研究流程：** 构造共用因子 → 批量分组回测（跨因子×跨指数×跨持仓期）→ 汇总对比


In [ ]:
import importlib
import sys
import warnings
from collections import defaultdict
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# 项目根目录自动探测
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "my_utils").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("未找到项目根目录：上级目录中缺少 my_utils/")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from my_utils.fun import read_day_data
from my_utils.rqdata import RqData
from 因子回测.alpha import backtest_timeseries_factor
from 因子回测.涨跌停情绪因子.timing_engine import (
    build_daily_sentiment_factors, prepare_stock_daily,
)

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", message="Glyph .* missing from current font")

START_DATE = date(2018, 1, 2)
END_DATE = date(2026, 7, 27)
DATA_SOURCE = "rq_stock_all_data"
WINDOW = 5
HORIZONS = (1, 3, 5, 10)
MIN_HISTORY = 252

FACTOR_COLUMNS = [
    "limit_up_ratio",
    "limit_down_ratio",
    "limit_up_next_ret",
    "limit_down_next_ret",
]
FACTOR_LABELS = {
    "limit_up_ratio": "涨停占比",
    "limit_down_ratio": "跌停占比",
    "limit_up_next_ret": "涨停次日收益",
    "limit_down_next_ret": "跌停次日收益",
}

BENCHMARK_CODES = {
    "hs300": "000300.XSHG",
    "zz500": "000905.XSHG",
    "zz1000": "000852.XSHG",
    "zz2000": "000906.XSHG",
    "cyb": "399006.XSHE",
    "kc50": "000688.XSHG",
}
BENCHMARK_LABELS = {
    "hs300": "沪深300", "zz500": "中证500", "zz1000": "中证1000",
    "zz2000": "中证2000", "cyb": "创业板指", "kc50": "科创50",
}
BENCHMARK_ORDER = ["hs300", "zz500", "zz1000", "zz2000", "cyb", "kc50"]


In [ ]:
# ===== 1. 构建情绪因子 =====
daily_fields = [
    "code", "trading_date", "close", "pre_close", "limit_up", "limit_down",
    "is_st", "is_suspended", "total_mv",
]
daily_raw = read_day_data(
    START_DATE, END_DATE, fields=daily_fields, file_path=DATA_SOURCE,
).sort(["code", "trading_date"])

prepared_daily, trading_calendar = prepare_stock_daily(daily_raw)
factor_daily = build_daily_sentiment_factors(
    prepared_daily, trading_calendar, window=WINDOW,
)

factor_data = (
    factor_daily.to_pandas()
    .assign(trading_date=lambda d: pd.to_datetime(d["trading_date"]))
    .sort_values("trading_date")
    .reset_index(drop=True)
)

# 由于因子计算需要 MIN_HISTORY 天预热，裁剪到有效范围
factor_data = factor_data.dropna(subset=FACTOR_COLUMNS, how="all").reset_index(drop=True)
print(f"因子数据范围: {factor_data['trading_date'].min()} ~ {factor_data['trading_date'].max()}")
print(f"有效交易日: {len(factor_data)}")
display(factor_data[["trading_date"] + FACTOR_COLUMNS].tail())


In [ ]:
# ===== 2. 获取多指数收益 =====
rq_client = RqData()
rq_returns = rq_client.get_return(
    list(BENCHMARK_CODES.values()), START_DATE, END_DATE,
)

# 从米筐 MultiIndex 拆成单指数表
normalized = (
    rq_returns.reset_index()[["order_book_id", "date", "return"]].copy()
)
normalized["date"] = pd.to_datetime(normalized["date"], errors="raise")
normalized["return"] = pd.to_numeric(normalized["return"], errors="raise").astype(float)

# 过滤 NaN/inf
bad_mask = ~np.isfinite(normalized["return"].to_numpy())
if bad_mask.any():
    bad_rows = normalized.loc[bad_mask]
    print(f"⚠ 跳过 {len(bad_rows)} 行 NaN/inf 指数收益")
    normalized = normalized.loc[~bad_mask].copy()

benchmark_returns = {}
for label, code in BENCHMARK_CODES.items():
    single = (
        normalized[normalized["order_book_id"] == code]
        [["date", "return"]]
        .rename(columns={"date": "trading_date", "return": "benchmark_ret"})
        .sort_values("trading_date")
        .reset_index(drop=True)
    )
    if single.empty:
        raise ValueError(f"指数 {label}({code}) 无有效收益数据")
    benchmark_returns[label] = single

# 统一到共同区间
common_start = max(d["trading_date"].min() for d in benchmark_returns.values())
common_end = min(d["trading_date"].max() for d in benchmark_returns.values())
for label in benchmark_returns:
    benchmark_returns[label] = (
        benchmark_returns[label]
        .query("trading_date >= @common_start and trading_date <= @common_end")
        .reset_index(drop=True)
    )
print(f"指数共同区间: {common_start.date()} ~ {common_end.date()}")
print(f"交易日数: {len(benchmark_returns['hs300'])}")


In [ ]:
# ===== 3. 批量回测与结果提取 =====

def extract_metrics(result, factor, benchmark, horizon):
    """从 backtest_timeseries_factor 结果中提取关键指标。"""
    perf = result["group_performance"]
    groups = [g for g in perf.index if g.startswith("G")]
    if len(groups) < 2:
        return None

    g1 = perf.loc["G1"]
    gq = perf.loc[groups[-1]]
    bm = perf.loc["买入持有基准"]

    cum_vals = [perf.loc[g, "累计收益"] for g in groups]
    monotonic_up = all(
        cum_vals[i] <= cum_vals[i + 1] for i in range(len(cum_vals) - 1)
    )

    return {
        "factor": factor,
        "benchmark": benchmark,
        "horizon": horizon,
        "G1_收益": g1["累计收益"],
        "Gq_收益": gq["累计收益"],
        "Gq-G1": gq["累计收益"] - g1["累计收益"],
        "G1_夏普": g1["夏普比率"],
        "Gq_夏普": gq["夏普比率"],
        "基准_收益": bm["累计收益"],
        "单调递增": monotonic_up,
    }


# 批量回测
rows = []
nav_cache = {}
total = len(FACTOR_COLUMNS) * len(BENCHMARK_ORDER) * len(HORIZONS)
count = 0

for factor in FACTOR_COLUMNS:
    for benchmark in BENCHMARK_ORDER:
        analysis_data = pd.merge(
            factor_data[["trading_date", factor]],
            benchmark_returns[benchmark],
            on="trading_date",
            how="inner",
        ).set_index("trading_date")
        analysis_data["ret_pct"] = analysis_data["benchmark_ret"] * 100

        for horizon in HORIZONS:
            count += 1
            if count % 10 == 0:
                print(f"  回测进度: {count}/{total} ({factor}, {benchmark}, {horizon}d)")

            result = backtest_timeseries_factor(
                analysis_data,
                factor_col=factor,
                index_ret_col="ret_pct",
                q=5,
                hold_period=horizon,
                plot=False,
                verbose=False,
            )

            metrics = extract_metrics(result, factor, benchmark, horizon)
            if metrics is not None:
                rows.append(metrics)

            nav_cache[(factor, benchmark, horizon)] = {
                "group_nav": result["group_nav"],
                "benchmark_nav": (1 + analysis_data["ret_pct"] / 100).cumprod(),
            }

print(f"\n批量回测完成！共 {len(rows)} 条有效记录，{len(nav_cache)} 组 NAV 已缓存")

summary = pd.DataFrame(rows)


# ========= 批量回测结束后绘制合成图 =========

# ----- 图 A：每个因子一张多面板净值图（行=持仓期，列=指数） -----
for factor in FACTOR_COLUMNS:
    fig, axes = plt.subplots(
        len(HORIZONS), len(BENCHMARK_ORDER),
        figsize=(3 * len(BENCHMARK_ORDER) + 2, 3 * len(HORIZONS) + 1),
        squeeze=False,
    )
    for hi, horizon in enumerate(HORIZONS):
        for bi, benchmark in enumerate(BENCHMARK_ORDER):
            ax = axes[hi][bi]
            key = (factor, benchmark, horizon)
            if key not in nav_cache:
                ax.set_visible(False)
                continue

            data = nav_cache[key]
            gn = data["group_nav"]
            bm_nav = data["benchmark_nav"]
            gn = gn / gn.iloc[0]
            bm_nav = bm_nav / bm_nav.iloc[0]

            for col in gn.columns:
                ax.plot(gn.index, gn[col], label=col, linewidth=0.8)
            ax.plot(bm_nav.index, bm_nav, color="black", linewidth=1.2, linestyle="--")

            if hi == 0:
                ax.set_title(BENCHMARK_LABELS[benchmark], fontsize=10)
            if bi == 0:
                ax.set_ylabel(f"{horizon}d", fontsize=9)
            ax.tick_params(labelsize=6)
            ax.grid(alpha=0.2)

    fig.suptitle(f"{FACTOR_LABELS[factor]}：各指数 × 各持仓期分组净值", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()

# ----- 图 B：Gq-G1 多空差热力图（每个持仓期一张） -----
for horizon in HORIZONS:
    fig, ax = plt.subplots(figsize=(7, 5))
    pivot = summary.query("horizon == @horizon").pivot_table(
        index="benchmark", columns="factor", values="Gq-G1",
    )
    im = ax.imshow(pivot.to_numpy(), cmap="RdYlGn", aspect="auto")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([FACTOR_LABELS[c] for c in pivot.columns], fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([BENCHMARK_LABELS[i] for i in pivot.index], fontsize=9)
    ax.set_title(f"Gq-G1 多空差（持仓 {horizon} 期）", fontsize=13)

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.iloc[i, j]
            ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(val) > pivot.abs().to_numpy().mean() else "black")
    fig.tight_layout()
    plt.show()

In [ ]:
# ===== 4. 结果汇总表 =====
# summary 和 nav_cache 已在 Cell 4 中生成

print("Gq-G1 多空差排序（跨指数均值）：")
top_combo = (
    summary.groupby(["factor", "horizon"])["Gq-G1"]
    .mean()
    .sort_values(ascending=False)
)
display(top_combo.to_frame("Gq-G1_均值").style.format("{:.2f}").background_gradient(cmap="RdYlGn"))

# 选单调性好的因子+持仓期
print("\n单调性统计（跨指数）")
mono_view = (
    summary.groupby(["factor", "horizon"])["单调递增"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "单调递增_指数数", "count": "总指数数"})
)
mono_view["单调性比率"] = mono_view["单调递增_指数数"] / mono_view["总指数数"]
display(mono_view.style.format({"单调性比率": "{:.0%}"}).background_gradient(cmap="RdYlGn", subset=["单调递增_指数数"]))

In [ ]:
# ===== 5. 选调：最优组合净值对比（使用 nav_cache） =====

ranked = summary.sort_values("Gq-G1", ascending=False)
print("Gq-G1 最优 TOP 10:")
display(ranked.head(10).style.format({
    "Gq-G1": "{:.2f}", "G1_收益": "{:.2f}", "Gq_收益": "{:.2f}",
}))

# TOP 4 放大看（单行多列）
N = 4
ncols = 4
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4), squeeze=False)

for idx, (_, row) in enumerate(ranked.head(N).iterrows()):
    key = (row["factor"], row["benchmark"], int(row["horizon"]))
    if key not in nav_cache:
        continue

    ax = axes[0][idx]
    data = nav_cache[key]
    gn = data["group_nav"] / data["group_nav"].iloc[0]
    bm = data["benchmark_nav"] / data["benchmark_nav"].iloc[0]

    for col in gn.columns:
        ax.plot(gn.index, gn[col], label=col, linewidth=1)
    ax.plot(bm.index, bm, color="black", linewidth=1.5, linestyle="--", label="基准")
    ax.set_title(f"{FACTOR_LABELS.get(row.factor, row.factor)} | {BENCHMARK_LABELS.get(row.benchmark, row.benchmark)} ({int(row.horizon)}d)", fontsize=10)
    ax.legend(fontsize=7, loc="best")
    ax.grid(alpha=0.3)

fig.suptitle("TOP 4 精选净值曲线", fontsize=14)
fig.tight_layout()
plt.show()